# 04 - Live Demo: Base vs Fine-tuned (Qwen2.5-1.5B + LoRA)
Runtime -> Change runtime type -> **T4 GPU**, then run all cells top to bottom.
Cell 3 asks you to upload `lora_adapter_150ex.zip` (zip of the adapter folder from your repo's `models/`).

In [ ]:
!pip -q install -U transformers peft accelerate gradio

In [ ]:
# 2. Upload and unzip the adapter
from google.colab import files
import zipfile, os, glob
up = files.upload()                      # pick lora_adapter_150ex.zip
zname = list(up.keys())[0]
os.makedirs("adapter", exist_ok=True)
zipfile.ZipFile(zname).extractall("adapter")
cfg = glob.glob("adapter/**/adapter_config.json", recursive=True)[0]
ADAPTER_DIR = os.path.dirname(cfg)
print("Adapter found in:", ADAPTER_DIR)

In [ ]:
# 3. Load base model on GPU (fp16) and attach the LoRA adapter
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16, device_map="auto")
model = PeftModel.from_pretrained(base, ADAPTER_DIR)
model.eval()
print("Ready on", next(model.parameters()).device)

In [ ]:
# 4. Answer function: same model, adapter ON (fine-tuned) vs adapter OFF (base)
import time

def answer(question, use_adapter, max_new_tokens=200):
    msgs = [{"role": "user", "content": question}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to(model.device)
    t0 = time.time()
    with torch.no_grad():
        if use_adapter:
            out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
        else:
            with model.disable_adapter():
                out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    dt = time.time() - t0
    reply = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    return f"{reply}\n\n[{dt:.1f}s, {len(reply.split())} words]"

print(answer("What is a decorator in Python?", True))

In [ ]:
# 5. Live side-by-side demo (prints a public link you can open or share)
import gradio as gr

def compare(q):
    return answer(q, False), answer(q, True)

with gr.Blocks(title="Base vs Fine-tuned") as demo:
    gr.Markdown("## Base Qwen2.5-1.5B vs LoRA fine-tuned on my lecture notes")
    q = gr.Textbox(label="Your question", placeholder="What is the difference between decorators and context managers?")
    btn = gr.Button("Ask both")
    with gr.Row():
        b = gr.Textbox(label="Base model", lines=12)
        f = gr.Textbox(label="Fine-tuned model", lines=12)
    btn.click(compare, q, [b, f])
    gr.Examples(["What is a decorator in Python?",
                 "What is database sharding?",
                 "What is the capital of France?"], q)

demo.launch(share=True)